# CVPR 2026 Children Gait End-to-End Solution

This notebook runs the complete solution used for the selected public submission.

Inputs required:

```text
/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge
/kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset
```

Recommended accelerator: Kaggle T4 GPU for the pose-TCN stage.


## Approach

The solution is intentionally split by task.

Track 2 is trained first with a compact pose temporal convolution model. This gives stable gait-subtype predictions for the nine Track 2 test patients.

Track 1 is then trained with a patient-level clinical feature ensemble. The features summarize joint angles, trunk posture, foot progression proxies, cadence proxies, frequency-domain ankle motion, symmetry, and stance/swing behavior across all views. The final classifier is an ensemble of Gradient Boosting, Extra Trees, and Random Forest multi-output models with per-label threshold tuning.

This structure worked better than trying to force one model to solve both tracks.


In [1]:

from pathlib import Path

COMP_ROOT = Path('/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge')
DATA_ROOT = Path('/kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset')
WORK_DIR = Path('/kaggle/working')

print('competition input:', COMP_ROOT.exists(), COMP_ROOT)
print('pose dataset:', DATA_ROOT.exists(), DATA_ROOT)
print('working directory:', WORK_DIR.exists(), WORK_DIR)


competition input: True /kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge
pose dataset: True /kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset
working directory: True /kaggle/working


## Stage 2: Pose TCN for Track 2

This stage builds fixed-length pose tensors from the raw JSON files, trains compact temporal convolution models, and writes `submission_stage2_pose_tcn.csv`. Stage 3 will reuse only the Track 2 rows from this file.


In [ ]:

from __future__ import annotations

import csv
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter, defaultdict
from datetime import datetime, UTC
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# =========================
# Config
# =========================

SEED = 123
T = 128
VIEWS = ["backward", "forward", "left", "right"]
N_JOINTS = 23
N_CHANNELS = 6  # bbox x/y, hip-centered x/y, score, mask
LOW_SCORE_THR = 0.20

EPOCHS_T1 = 70
EPOCHS_T2 = 90
BATCH_SIZE_T1 = 16
BATCH_SIZE_T2 = 8
LR = 2e-3
WEIGHT_DECAY = 2e-3
DROPOUT = 0.35

KAGGLE_TRACK1 = Path("/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track1_train.json")
KAGGLE_TRACK2 = Path("/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track2_train.json")
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset")

PATH1 = KAGGLE_TRACK1
PATH2 = KAGGLE_TRACK2
DATA_ROOT = KAGGLE_DATA_ROOT

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("kaggle_working")
RUN_ID = datetime.now(UTC).strftime("stage2_%Y%m%d_%H%M%S")
TABLE_DIR = WORK_DIR / "tables" / RUN_ID
VERSION_DIR = WORK_DIR / "versioning" / RUN_ID
MODEL_DIR = VERSION_DIR / "models"

T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]
LABEL_COLS = [f"L{i}" for i in range(1, 18)] + [f"R{i}" for i in range(1, 18)]
SUBMISSION_COLUMNS = (
    ["ID"]
    + [f"L{i}" for i in range(1, 18)]
    + [f"R{i}" for i in range(1, 18)]
    + ["Total", "Left_gait_subtype", "Right_gait_subtype"]
)
CLASS_NAMES = ["WNL", "type1", "type2", "type3", "type4"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

VIEW_RE = re.compile(r"_(forward|backward|left|right)_", re.IGNORECASE)


# =========================
# Utilities
# =========================

def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def section(title: str) -> None:
    print("\n" + "=" * 80, flush=True)
    print(title, flush=True)
    print("=" * 80, flush=True)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs() -> None:
    for d in [TABLE_DIR, VERSION_DIR, MODEL_DIR]:
        d.mkdir(parents=True, exist_ok=True)


def dump_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def parse_view(sequence_dir: Path) -> str:
    m = VIEW_RE.search(sequence_dir.name)
    return m.group(1).lower() if m else "unknown"


def parse_patient_id(sequence_dir: Path) -> int:
    digits = re.sub(r"\D", "", sequence_dir.parent.name)
    return int(digits)


def sample_evenly(items: list[Path], n: int) -> list[Path]:
    if not items:
        return []
    if len(items) <= n:
        return items
    idx = np.linspace(0, len(items) - 1, n).round().astype(int)
    return [items[i] for i in idx]


def flatten_track1(track1: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for item in track1:
        row = {"patient_id": int(item["patient_id"])}
        for i in range(1, 18):
            row[f"L{i}"] = int(item["left"][str(i)])
            row[f"R{i}"] = int(item["right"][str(i)])
        row["Total"] = sum(row[c] for c in LABEL_COLS)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)


def flatten_track2(track2: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for item in track2:
        rows.append(
            {
                "patient_id": int(item["patient_id"]),
                "Left_gait_subtype": item["left"]["gait_subtype"],
                "Right_gait_subtype": item["right"]["gait_subtype"],
                "left_id": CLASS_TO_ID[item["left"]["gait_subtype"]],
                "right_id": CLASS_TO_ID[item["right"]["gait_subtype"]],
            }
        )
    return pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)


def build_index(data_root: Path) -> pd.DataFrame:
    rows = []
    sequence_dirs = sorted({p.parent for p in data_root.rglob("frame_*.json")})
    for i, seq_dir in enumerate(sequence_dirs, start=1):
        files = sorted(seq_dir.glob("frame_*.json"))
        first = load_json(files[0]) if files else {}
        vi = first.get("video_info", {}) if isinstance(first, dict) else {}
        rows.append(
            {
                "patient_id": parse_patient_id(seq_dir),
                "view": parse_view(seq_dir),
                "sequence_dir": str(seq_dir),
                "frame_count": len(files),
                "fps": vi.get("fps", np.nan),
                "total_frames": vi.get("total_frames", np.nan),
                "video_name": vi.get("video_name", ""),
            }
        )
        if i % 250 == 0:
            log(f"Indexed {i}/{len(sequence_dirs)} sequence dirs")
    return pd.DataFrame(rows)


# =========================
# Tensor building
# =========================

def frame_to_tensor(path: Path) -> np.ndarray:
    data = load_json(path)
    vi = data.get("video_info", {})
    width = float(vi.get("width", 1920))
    height = float(vi.get("height", 1080))
    insts = data.get("instance_info", [])
    out = np.zeros((N_JOINTS, N_CHANNELS), dtype=np.float32)
    if not insts:
        return out

    inst = insts[0]
    kp = np.asarray(inst.get("keypoints", []), dtype=np.float32)
    scores = np.asarray(inst.get("keypoint_scores", []), dtype=np.float32)
    if kp.ndim != 2 or kp.shape[0] < N_JOINTS or kp.shape[1] < 2:
        return out
    if scores.shape[0] < kp.shape[0]:
        scores = np.ones((kp.shape[0],), dtype=np.float32)

    bbox = inst.get("gt_bbox_xywh_px")
    if bbox and len(bbox) >= 4:
        bx, by, bw, bh = [float(v) for v in bbox[:4]]
        if bw <= 1 or bh <= 1:
            bx, by, bw, bh = 0.0, 0.0, width, height
    else:
        bx, by, bw, bh = 0.0, 0.0, width, height

    coords = kp[:N_JOINTS, :2].copy()
    bbox_xy = coords.copy()
    bbox_xy[:, 0] = (bbox_xy[:, 0] - bx) / max(bw, 1.0)
    bbox_xy[:, 1] = (bbox_xy[:, 1] - by) / max(bh, 1.0)

    hip_mid = (bbox_xy[11] + bbox_xy[12]) / 2.0
    centered = bbox_xy - hip_mid[None, :]
    scale = max(np.linalg.norm(bbox_xy[5] - bbox_xy[11]), np.linalg.norm(bbox_xy[6] - bbox_xy[12]), 0.05)
    centered = centered / scale

    sc = np.clip(scores[:N_JOINTS], 0.0, 1.5)
    mask = (sc >= LOW_SCORE_THR).astype(np.float32)

    out[:, 0:2] = bbox_xy
    out[:, 2:4] = centered
    out[:, 4] = sc
    out[:, 5] = mask
    return out


def collect_view_files(index_df: pd.DataFrame, patient_id: int, view: str) -> list[Path]:
    sub = index_df[(index_df["patient_id"] == patient_id) & (index_df["view"] == view)]
    all_files: list[Path] = []
    for seq_dir in sub["sequence_dir"].tolist():
        all_files.extend(sorted(Path(seq_dir).glob("frame_*.json")))
    return all_files


def build_pose_tensor(index_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    cache_path = TABLE_DIR / f"pose_tensor_T{T}.npz"
    if cache_path.exists():
        log(f"Loading cached tensor: {cache_path}")
        data = np.load(cache_path)
        return data["X"], data["patient_ids"]

    patient_ids = np.array(sorted(index_df["patient_id"].unique()), dtype=np.int32)
    X = np.zeros((len(patient_ids), len(VIEWS), T, N_JOINTS, N_CHANNELS), dtype=np.float16)
    for pi, pid in enumerate(patient_ids):
        for vi, view in enumerate(VIEWS):
            files = collect_view_files(index_df, int(pid), view)
            chosen = sample_evenly(files, T)
            if not chosen:
                continue
            frames = [frame_to_tensor(p) for p in chosen]
            # If fewer than T, pad by repeating the last valid frame.
            while len(frames) < T:
                frames.append(frames[-1].copy())
            X[pi, vi] = np.stack(frames[:T]).astype(np.float16)
        if (pi + 1) % 10 == 0 or pi + 1 == len(patient_ids):
            log(f"Built pose tensor for {pi + 1}/{len(patient_ids)} patients")

    np.savez_compressed(cache_path, X=X, patient_ids=patient_ids)
    log(f"Saved tensor: {cache_path} shape={X.shape}")
    return X, patient_ids


# =========================
# Models
# =========================

class PoseEncoder(nn.Module):
    def __init__(self, dropout: float = DROPOUT):
        super().__init__()
        in_ch = N_JOINTS * N_CHANNELS
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(128, 192, kernel_size=3, padding=1),
            nn.BatchNorm1d(192),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: B,V,T,J,C
        b, v, t, j, c = x.shape
        x = x.reshape(b * v, t, j * c).permute(0, 2, 1)
        h = self.net(x)
        avg = h.mean(dim=-1)
        mx = h.amax(dim=-1)
        h = torch.cat([avg, mx], dim=1).reshape(b, v, -1)
        return h.reshape(b, -1)


class Track1Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = PoseEncoder()
        self.head = nn.Sequential(
            nn.Linear(len(VIEWS) * 192 * 2, 384),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(384, 34),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))


class Track2Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = PoseEncoder(dropout=0.45)
        self.shared = nn.Sequential(
            nn.Linear(len(VIEWS) * 192 * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.45),
        )
        self.left = nn.Linear(256, len(CLASS_NAMES))
        self.right = nn.Linear(256, len(CLASS_NAMES))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.shared(self.encoder(x))
        return self.left(h), self.right(h)


def device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def make_loader(X: np.ndarray, y: np.ndarray | list[np.ndarray], batch_size: int, shuffle: bool) -> DataLoader:
    tx = torch.tensor(X, dtype=torch.float32)
    if isinstance(y, list):
        tensors = [tx] + [torch.tensor(v) for v in y]
    else:
        tensors = [tx, torch.tensor(y)]
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle, drop_last=False)


def s1_metric(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    acc = float((y_true == y_pred).mean())
    rmse = float(np.sqrt(np.mean((y_pred.sum(axis=1) - y_true.sum(axis=1)) ** 2)))
    score = float((acc + 1.0 - rmse / 34.0) / 2.0)
    return {"accuracy": acc, "rmse": rmse, "score": score}


def s2_metric(lt: np.ndarray, rt: np.ndarray, lp: np.ndarray, rp: np.ndarray) -> dict[str, float]:
    acc = float(np.mean(np.concatenate([lt == lp, rt == rp])))
    f1_l = float(f1_score(lt, lp, average="macro", zero_division=0))
    f1_r = float(f1_score(rt, rp, average="macro", zero_division=0))
    f1 = (f1_l + f1_r) / 2.0
    return {"accuracy": acc, "macro_f1": f1, "score": (acc + f1) / 2.0}


def tune_thresholds(y_true: np.ndarray, prob: np.ndarray) -> np.ndarray:
    thresholds = np.full(prob.shape[1], 0.5, dtype=np.float32)
    for c in range(prob.shape[1]):
        best_t, best_acc = 0.5, -1.0
        for t in np.linspace(0.2, 0.8, 25):
            acc = accuracy_score(y_true[:, c], (prob[:, c] >= t).astype(int))
            if acc > best_acc:
                best_t, best_acc = float(t), float(acc)
        thresholds[c] = 0.75 * best_t + 0.25 * 0.5
    return thresholds


# =========================
# Training
# =========================

def train_track1_fold(X_train, y_train, X_valid, y_valid, fold: int) -> tuple[np.ndarray, Track1Net]:
    dev = device()
    model = Track1Net().to(dev)
    pos = np.clip(y_train.sum(axis=0), 1, None)
    neg = np.clip(len(y_train) - y_train.sum(axis=0), 1, None)
    pos_weight = torch.tensor(np.clip(neg / pos, 0.5, 6.0), dtype=torch.float32, device=dev)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_loader = make_loader(X_train, y_train.astype(np.float32), BATCH_SIZE_T1, True)
    valid_loader = make_loader(X_valid, y_valid.astype(np.float32), BATCH_SIZE_T1, False)

    best_score = -999.0
    best_state = None
    best_prob = None

    for epoch in range(1, EPOCHS_T1 + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(dev)
            yb = yb.to(dev)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = bce(logits, yb)
            # Soft total loss helps RMSE without forcing hard labels.
            loss = loss + 0.025 * ((torch.sigmoid(logits).sum(dim=1) - yb.sum(dim=1)) ** 2).mean() / 34.0
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            losses.append(float(loss.detach().cpu()))

        if epoch % 5 == 0 or epoch == EPOCHS_T1:
            model.eval()
            probs = []
            with torch.no_grad():
                for xb, _ in valid_loader:
                    logits = model(xb.to(dev))
                    probs.append(torch.sigmoid(logits).cpu().numpy())
            prob = np.vstack(probs)
            pred = (prob >= 0.5).astype(int)
            score = s1_metric(y_valid.astype(int), pred)["score"]
            if score > best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_prob = prob.copy()
        if epoch in [1, 20, 40, 60, EPOCHS_T1]:
            log(f"T1 fold {fold} epoch {epoch}/{EPOCHS_T1} loss={np.mean(losses):.5f} best_s1={best_score:.5f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_prob if best_prob is not None else np.zeros_like(y_valid, dtype=np.float32), model


def train_track1_oof(X_all: np.ndarray, patient_ids: np.ndarray, track1_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, dict[str, float], np.ndarray]:
    pid_to_idx = {int(pid): i for i, pid in enumerate(patient_ids)}
    train_pids = track1_df["patient_id"].astype(int).tolist()
    idx = np.array([pid_to_idx[p] for p in train_pids], dtype=int)
    X = X_all[idx]
    y = track1_df[LABEL_COLS].values.astype(np.float32)

    oof = np.zeros_like(y, dtype=np.float32)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (tr, va) in enumerate(kf.split(X), start=1):
        prob, _ = train_track1_fold(X[tr], y[tr], X[va], y[va], fold)
        oof[va] = prob
        fold_pred = (prob >= 0.5).astype(int)
        score = s1_metric(y[va].astype(int), fold_pred)
        log(f"T1 fold {fold} final 0.5: S1={score['score']:.5f} acc={score['accuracy']:.5f} rmse={score['rmse']:.5f}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    thresholds = tune_thresholds(y.astype(int), oof)
    tuned = (oof >= thresholds[None, :]).astype(int)
    score = s1_metric(y.astype(int), tuned)
    log(f"T1 OOF tuned: S1={score['score']:.5f} acc={score['accuracy']:.5f} rmse={score['rmse']:.5f}")
    return oof, y, score, thresholds


def train_track1_final(X_all: np.ndarray, patient_ids: np.ndarray, track1_df: pd.DataFrame) -> Track1Net:
    pid_to_idx = {int(pid): i for i, pid in enumerate(patient_ids)}
    idx = np.array([pid_to_idx[int(p)] for p in track1_df["patient_id"]], dtype=int)
    X = X_all[idx]
    y = track1_df[LABEL_COLS].values.astype(np.float32)
    # Train final with all data. Use validation equal to train just to reuse function.
    _, model = train_track1_fold(X, y, X, y, fold=99)
    return model


def class_weights(values: np.ndarray) -> torch.Tensor:
    counts = np.bincount(values, minlength=len(CLASS_NAMES)).astype(np.float32)
    weights = counts.sum() / np.clip(counts, 1, None)
    weights = weights / weights.mean()
    return torch.tensor(np.clip(weights, 0.5, 5.0), dtype=torch.float32)


def train_track2_fold(X_train, yl_train, yr_train, X_valid, yl_valid, yr_valid, fold: int) -> tuple[np.ndarray, np.ndarray, Track2Net]:
    dev = device()
    model = Track2Net().to(dev)
    ce_l = nn.CrossEntropyLoss(weight=class_weights(yl_train).to(dev))
    ce_r = nn.CrossEntropyLoss(weight=class_weights(yr_train).to(dev))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_loader = make_loader(X_train, [yl_train.astype(np.int64), yr_train.astype(np.int64)], BATCH_SIZE_T2, True)
    valid_loader = make_loader(X_valid, [yl_valid.astype(np.int64), yr_valid.astype(np.int64)], BATCH_SIZE_T2, False)

    best_score = -999.0
    best_state = None
    best_l = None
    best_r = None
    for epoch in range(1, EPOCHS_T2 + 1):
        model.train()
        losses = []
        for xb, ylb, yrb in train_loader:
            xb = xb.to(dev)
            ylb = ylb.to(dev)
            yrb = yrb.to(dev)
            opt.zero_grad(set_to_none=True)
            log_l, log_r = model(xb)
            loss = ce_l(log_l, ylb) + ce_r(log_r, yrb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            losses.append(float(loss.detach().cpu()))

        if epoch % 5 == 0 or epoch == EPOCHS_T2:
            model.eval()
            lp, rp = [], []
            with torch.no_grad():
                for xb, _, _ in valid_loader:
                    log_l, log_r = model(xb.to(dev))
                    lp.append(torch.softmax(log_l, dim=1).cpu().numpy())
                    rp.append(torch.softmax(log_r, dim=1).cpu().numpy())
            lp_arr = np.vstack(lp)
            rp_arr = np.vstack(rp)
            pred_l = lp_arr.argmax(axis=1)
            pred_r = rp_arr.argmax(axis=1)
            score = s2_metric(yl_valid, yr_valid, pred_l, pred_r)["score"]
            if score > best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_l = lp_arr.copy()
                best_r = rp_arr.copy()
        if epoch in [1, 30, 60, EPOCHS_T2]:
            log(f"T2 fold {fold} epoch {epoch}/{EPOCHS_T2} loss={np.mean(losses):.5f} best_s2={best_score:.5f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_l, best_r, model


def train_track2_oof(X_all: np.ndarray, patient_ids: np.ndarray, track2_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, dict[str, float]]:
    pid_to_idx = {int(pid): i for i, pid in enumerate(patient_ids)}
    idx = np.array([pid_to_idx[int(p)] for p in track2_df["patient_id"]], dtype=int)
    X = X_all[idx]
    yl = track2_df["left_id"].values.astype(np.int64)
    yr = track2_df["right_id"].values.astype(np.int64)

    oof_l = np.zeros((len(X), len(CLASS_NAMES)), dtype=np.float32)
    oof_r = np.zeros((len(X), len(CLASS_NAMES)), dtype=np.float32)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (tr, va) in enumerate(kf.split(X), start=1):
        pl, pr, _ = train_track2_fold(X[tr], yl[tr], yr[tr], X[va], yl[va], yr[va], fold)
        oof_l[va] = pl
        oof_r[va] = pr
        fold_score = s2_metric(yl[va], yr[va], pl.argmax(axis=1), pr.argmax(axis=1))
        log(f"T2 fold {fold}: S2={fold_score['score']:.5f} acc={fold_score['accuracy']:.5f} f1={fold_score['macro_f1']:.5f}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    score = s2_metric(yl, yr, oof_l.argmax(axis=1), oof_r.argmax(axis=1))
    log(f"T2 OOF: S2={score['score']:.5f} acc={score['accuracy']:.5f} f1={score['macro_f1']:.5f}")
    return oof_l, oof_r, yl, yr, score


def train_track2_final(X_all: np.ndarray, patient_ids: np.ndarray, track2_df: pd.DataFrame) -> Track2Net:
    pid_to_idx = {int(pid): i for i, pid in enumerate(patient_ids)}
    idx = np.array([pid_to_idx[int(p)] for p in track2_df["patient_id"]], dtype=int)
    X = X_all[idx]
    yl = track2_df["left_id"].values.astype(np.int64)
    yr = track2_df["right_id"].values.astype(np.int64)
    _, _, model = train_track2_fold(X, yl, yr, X, yl, yr, fold=99)
    return model


def predict_track1(model: Track1Net, X: np.ndarray, thresholds: np.ndarray, shift: float = 0.0) -> np.ndarray:
    dev = device()
    model.eval()
    loader = make_loader(X, np.zeros((len(X), 34), dtype=np.float32), BATCH_SIZE_T1, False)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            probs.append(torch.sigmoid(model(xb.to(dev))).cpu().numpy())
    prob = np.vstack(probs)
    th = np.clip(thresholds + shift, 0.03, 0.97)
    return (prob >= th[None, :]).astype(int)


def predict_track2(model: Track2Net, X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    dev = device()
    model.eval()
    loader = make_loader(X, [np.zeros(len(X), dtype=np.int64), np.zeros(len(X), dtype=np.int64)], BATCH_SIZE_T2, False)
    lp, rp = [], []
    with torch.no_grad():
        for xb, _, _ in loader:
            log_l, log_r = model(xb.to(dev))
            lp.append(torch.softmax(log_l, dim=1).cpu().numpy())
            rp.append(torch.softmax(log_r, dim=1).cpu().numpy())
    l = np.vstack(lp).argmax(axis=1)
    r = np.vstack(rp).argmax(axis=1)
    return l, r


def make_submission(t1_pred: np.ndarray, t2_left_ids: np.ndarray, t2_right_ids: np.ndarray, out_path: Path) -> pd.DataFrame:
    rows = []
    for pid, labels in zip(T1_TEST_IDS, t1_pred.astype(int)):
        labels = labels.tolist()
        rows.append([f"track1-{pid}", *labels[:17], *labels[17:], int(sum(labels)), -1, -1])
    for pid, l, r in zip(T2_TEST_IDS, t2_left_ids, t2_right_ids):
        rows.append([f"track2-{pid}", *([-1] * 34), -1, ID_TO_CLASS[int(l)], ID_TO_CLASS[int(r)]])
    df = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)
    assert df.shape == (25, 38)
    df.to_csv(out_path, index=False)
    log(f"Wrote {out_path}")
    print(df.to_string(index=False), flush=True)
    return df


def main() -> None:
    start = time.time()
    seed_everything()
    ensure_dirs()
    section("Stage 2 Pose TCN")
    log(f"RUN_ID={RUN_ID}")
    log(f"device={device()}")
    log(f"PATH1={PATH1}")
    log(f"PATH2={PATH2}")
    log(f"DATA_ROOT={DATA_ROOT}")

    config = {
        "run_id": RUN_ID,
        "seed": SEED,
        "T": T,
        "views": VIEWS,
        "n_joints": N_JOINTS,
        "n_channels": N_CHANNELS,
        "epochs_t1": EPOCHS_T1,
        "epochs_t2": EPOCHS_T2,
        "batch_size_t1": BATCH_SIZE_T1,
        "batch_size_t2": BATCH_SIZE_T2,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
    }
    dump_json(config, VERSION_DIR / "run_config.json")

    section("Load labels and index")
    track1_df = flatten_track1(load_json(PATH1))
    track2_df = flatten_track2(load_json(PATH2))
    index_df = build_index(DATA_ROOT)
    index_df.to_csv(TABLE_DIR / "dataset_index.csv", index=False)
    track1_df.to_csv(TABLE_DIR / "track1_flat.csv", index=False)
    track2_df.to_csv(TABLE_DIR / "track2_flat.csv", index=False)
    log(f"index={index_df.shape}, patients={index_df.patient_id.nunique()}, frames={int(index_df.frame_count.sum())}")

    section("Build pose tensor")
    X_all, patient_ids = build_pose_tensor(index_df)
    log(f"X_all={X_all.shape}, dtype={X_all.dtype}")

    section("Track 1 OOF and final")
    oof_t1, y1, t1_score, thresholds = train_track1_oof(X_all, patient_ids, track1_df)
    pd.DataFrame(oof_t1, columns=LABEL_COLS).assign(patient_id=track1_df["patient_id"].values).to_csv(TABLE_DIR / "stage2_track1_oof_prob.csv", index=False)
    pd.DataFrame({"label": LABEL_COLS, "threshold": thresholds}).to_csv(TABLE_DIR / "stage2_track1_thresholds.csv", index=False)
    t1_model = train_track1_final(X_all, patient_ids, track1_df)
    torch.save(t1_model.state_dict(), MODEL_DIR / "track1_tcn.pt")

    section("Track 2 OOF and final")
    oof_l, oof_r, yl, yr, t2_score = train_track2_oof(X_all, patient_ids, track2_df)
    t2_oof_df = pd.DataFrame(
        {
            "patient_id": track2_df["patient_id"].values,
            "left_true": [ID_TO_CLASS[int(v)] for v in yl],
            "right_true": [ID_TO_CLASS[int(v)] for v in yr],
            "left_pred": [ID_TO_CLASS[int(v)] for v in oof_l.argmax(axis=1)],
            "right_pred": [ID_TO_CLASS[int(v)] for v in oof_r.argmax(axis=1)],
        }
    )
    t2_oof_df.to_csv(TABLE_DIR / "stage2_track2_oof_pred.csv", index=False)
    t2_model = train_track2_final(X_all, patient_ids, track2_df)
    torch.save(t2_model.state_dict(), MODEL_DIR / "track2_tcn.pt")

    section("Create submissions")
    pid_to_idx = {int(pid): i for i, pid in enumerate(patient_ids)}
    idx_t1 = np.array([pid_to_idx[p] for p in T1_TEST_IDS], dtype=int)
    idx_t2 = np.array([pid_to_idx[p] for p in T2_TEST_IDS], dtype=int)
    t1_pred = predict_track1(t1_model, X_all[idx_t1], thresholds, shift=0.0)
    t2_l, t2_r = predict_track2(t2_model, X_all[idx_t2])
    stage2 = make_submission(t1_pred, t2_l, t2_r, WORK_DIR / "submission_stage2_pose_tcn.csv")

    # Hybrid candidates using current known best Stage 1 threshold file if present.
    best_stage1_path = WORK_DIR / "submission_stage1_t1_lesspos_0125.csv"
    if best_stage1_path.exists():
        best_stage1 = pd.read_csv(best_stage1_path)
        hybrid1 = stage2.copy()
        # Use Stage 1 best Track 2, Stage 2 Track 1.
        hybrid1.loc[hybrid1["ID"].str.startswith("track2-"), :] = best_stage1.loc[best_stage1["ID"].str.startswith("track2-"), :].values
        hybrid1.to_csv(WORK_DIR / "submission_stage2_t1_stage1best_t2.csv", index=False)
        log(f"Wrote {WORK_DIR / 'submission_stage2_t1_stage1best_t2.csv'}")

        hybrid2 = best_stage1.copy()
        # Use Stage 1 best Track 1, Stage 2 Track 2.
        hybrid2.loc[hybrid2["ID"].str.startswith("track2-"), :] = stage2.loc[stage2["ID"].str.startswith("track2-"), :].values
        hybrid2.to_csv(WORK_DIR / "submission_stage1best_t1_stage2_t2.csv", index=False)
        log(f"Wrote {WORK_DIR / 'submission_stage1best_t1_stage2_t2.csv'}")
    else:
        log("No Stage 1 best file found; skipped hybrid submissions")

    summary = {
        "run_id": RUN_ID,
        "elapsed_minutes": (time.time() - start) / 60.0,
        "device": str(device()),
        "dataset": {
            "patients": int(index_df.patient_id.nunique()),
            "sequences": int(len(index_df)),
            "frames": int(index_df.frame_count.sum()),
        },
        "cv": {
            "track1": t1_score,
            "track2": t2_score,
            "mean_proxy": float((t1_score["score"] + t2_score["score"]) / 2.0),
        },
        "outputs": {
            "stage2": str(WORK_DIR / "submission_stage2_pose_tcn.csv"),
            "stage2_t1_stage1best_t2": str(WORK_DIR / "submission_stage2_t1_stage1best_t2.csv"),
            "stage1best_t1_stage2_t2": str(WORK_DIR / "submission_stage1best_t1_stage2_t2.csv"),
        },
    }
    dump_json(summary, VERSION_DIR / "run_summary.json")
    log(f"Saved summary: {VERSION_DIR / 'run_summary.json'}")
    log(f"Elapsed minutes: {(time.time() - start) / 60.0:.2f}")
    section("Done")


if __name__ == "__main__":
    main()



Stage 2 Pose TCN
[21:36:38] RUN_ID=stage2_20260428_213638
[21:36:38] device=cuda
[21:36:38] PATH1=/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track1_train.json
[21:36:38] PATH2=/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track2_train.json
[21:36:38] DATA_ROOT=/kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset

Load labels and index
[21:50:31] Indexed 250/1185 sequence dirs
[21:50:34] Indexed 500/1185 sequence dirs
[21:50:37] Indexed 750/1185 sequence dirs
[21:50:39] Indexed 1000/1185 sequence dirs
[21:50:41] index=(1185, 7), patients=110, frames=339236

Build pose tensor
[21:51:07] Built pose tensor for 10/110 patients
[21:51:34] Built pose tensor for 20/110 patients
[21:52:04] Built pose tensor for 30/110 patients
[21:52:33] Built pose tensor for 40/110 patients
[21:53:03] Built pose tensor for 50/110 patients
[21:53:33] Built pose tensor for 60/110 patients
[21:54:02] Built pose tensor for 70/110 patients
[21:54:32] Bui

## Stage 3: Clinical Feature Ensemble for Track 1

This stage extracts richer patient-level gait features, trains the final Track 1 ensemble, takes Track 2 rows from the Stage 2 output, and writes the selected submission file.

```


In [3]:

from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler

try:
    import joblib
except Exception:
    joblib = None


# =========================
# Config
# =========================

SEED = 2026
MAX_FRAMES_PER_VIEW = 500
CV_SEEDS = [42, 123, 456, 789, 321]
N_SPLITS = 5
N_JOBS = -1

KAGGLE_COMP_DIR = Path("/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge")
LOCAL_COMP_DIR = Path("data/kaggle_hosted")
KAGGLE_TRACK1 = KAGGLE_COMP_DIR / "track1_train.json"
KAGGLE_TRACK2 = KAGGLE_COMP_DIR / "track2_train.json"
LOCAL_TRACK1 = LOCAL_COMP_DIR / "track1_train.json"
LOCAL_TRACK2 = LOCAL_COMP_DIR / "track2_train.json"
PATH1 = KAGGLE_TRACK1 if KAGGLE_TRACK1.exists() else LOCAL_TRACK1
PATH2 = KAGGLE_TRACK2 if KAGGLE_TRACK2.exists() else LOCAL_TRACK2

KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset")
LOCAL_DATA_ROOT = Path("data/full_dataset/dataset")
DEFAULT_DATA_ROOT = KAGGLE_DATA_ROOT if KAGGLE_DATA_ROOT.exists() else LOCAL_DATA_ROOT
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("kaggle_working")
RUN_ID = datetime.now(UTC).strftime("stage3_cgc_%Y%m%d_%H%M%S")
TABLE_DIR = WORK_DIR / "tables" / RUN_ID
VERSION_DIR = WORK_DIR / "versioning" / RUN_ID
MODEL_DIR = VERSION_DIR / "models"

T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]
LABEL_COLS = [f"L{i}" for i in range(1, 18)] + [f"R{i}" for i in range(1, 18)]
SUBMISSION_COLUMNS = (
    ["ID"]
    + [f"L{i}" for i in range(1, 18)]
    + [f"R{i}" for i in range(1, 18)]
    + ["Total", "Left_gait_subtype", "Right_gait_subtype"]
)
VIEWS = ["backward", "forward", "left", "right"]
GAIT_IDXS = [5, 6, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
VIEW_RE = re.compile(r"_(forward|backward|left|right)_", re.IGNORECASE)


# =========================
# Utilities
# =========================

def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def section(title: str) -> None:
    print("\n" + "=" * 80, flush=True)
    print(title, flush=True)
    print("=" * 80, flush=True)


def ensure_dirs() -> None:
    for d in [TABLE_DIR, VERSION_DIR, MODEL_DIR]:
        d.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def dump_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def find_data_root() -> Path:
    if DEFAULT_DATA_ROOT.exists():
        return DEFAULT_DATA_ROOT
    for d in Path("/kaggle/input").rglob("dataset"):
        if d.is_dir() and any(d.glob("*/**/frame_*.json")):
            return d
    return DEFAULT_DATA_ROOT


def parse_view(seq_dir: Path) -> str:
    m = VIEW_RE.search(seq_dir.name)
    if m:
        return m.group(1).lower()
    lower = seq_dir.name.lower()
    for view in VIEWS:
        if view in lower:
            return view
    return "unknown"


def parse_patient_id(seq_dir: Path) -> int:
    digits = re.sub(r"\D", "", seq_dir.parent.name)
    return int(digits)


def sample_evenly(files: list[Path], max_count: int) -> list[Path]:
    if len(files) <= max_count:
        return files
    idx = np.linspace(0, len(files) - 1, max_count).round().astype(int)
    return [files[int(i)] for i in idx]


def flatten_track1(track1: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for item in track1:
        row = {"patient_id": int(item["patient_id"])}
        for i in range(1, 18):
            row[f"L{i}"] = int(item["left"][str(i)])
            row[f"R{i}"] = int(item["right"][str(i)])
        row["Total"] = int(sum(row[c] for c in LABEL_COLS))
        rows.append(row)
    return pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)


def build_patient_files(data_root: Path) -> dict[int, dict[str, list[Path]]]:
    patient_files: dict[int, dict[str, list[Path]]] = {}
    sequence_dirs = sorted({p.parent for p in data_root.rglob("frame_*.json")})
    for i, seq_dir in enumerate(sequence_dirs, start=1):
        try:
            pid = parse_patient_id(seq_dir)
        except Exception:
            continue
        view = parse_view(seq_dir)
        if view not in VIEWS:
            continue
        files = sorted(seq_dir.glob("frame_*.json"))
        patient_files.setdefault(pid, {v: [] for v in VIEWS})
        patient_files[pid][view].extend(files)
        if i % 250 == 0:
            log(f"Indexed {i}/{len(sequence_dirs)} sequence dirs")
    return patient_files


# =========================
# Pose feature extraction
# =========================

def load_kpts(files: list[Path], max_frames: int = MAX_FRAMES_PER_VIEW) -> tuple[np.ndarray | None, float]:
    chosen = sample_evenly(sorted(files), max_frames)
    rows = []
    fps = 30.0
    for fp in chosen:
        try:
            data = load_json(fp)
            fps = float(data.get("video_info", {}).get("fps", fps) or fps)
            inst = data.get("instance_info", [])
            if not inst:
                continue
            kp = np.asarray(inst[0].get("keypoints", []), dtype=np.float32)
            if kp.ndim == 2 and kp.shape[0] >= 23 and kp.shape[1] >= 2:
                rows.append(kp[:23, :2])
        except Exception:
            continue
    if not rows:
        return None, fps
    return np.stack(rows).astype(np.float32), fps


def ang3(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8
    cosv = np.sum(ba * bc, axis=-1) / denom
    return np.degrees(np.arccos(np.clip(cosv, -1.0, 1.0)))


def stats(arr: np.ndarray, prefix: str) -> dict[str, float]:
    x = np.asarray(arr, dtype=np.float32).reshape(-1)
    x = x[np.isfinite(x)]
    if x.size == 0:
        x = np.array([0.0], dtype=np.float32)
    return {
        f"{prefix}_m": float(np.mean(x)),
        f"{prefix}_s": float(np.std(x)),
        f"{prefix}_mn": float(np.min(x)),
        f"{prefix}_mx": float(np.max(x)),
        f"{prefix}_r": float(np.ptp(x)),
        f"{prefix}_md": float(np.median(x)),
        f"{prefix}_p10": float(np.percentile(x, 10)),
        f"{prefix}_p25": float(np.percentile(x, 25)),
        f"{prefix}_p75": float(np.percentile(x, 75)),
        f"{prefix}_p90": float(np.percentile(x, 90)),
    }


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if len(a) < 3 or len(b) < 3:
        return 0.0
    a = a - np.nanmean(a)
    b = b - np.nanmean(b)
    val = float(np.corrcoef(a, b)[0, 1])
    return val if np.isfinite(val) else 0.0


def interp_seq(seq: np.ndarray, target: int = 60) -> np.ndarray:
    seq = np.asarray(seq, dtype=np.float32).reshape(-1)
    seq = seq[np.isfinite(seq)]
    if len(seq) == 0:
        return np.zeros(target, dtype=np.float32)
    if len(seq) == 1:
        return np.full(target, float(seq[0]), dtype=np.float32)
    x = np.linspace(0, 1, len(seq))
    xn = np.linspace(0, 1, target)
    return interp1d(x, seq, kind="linear", fill_value="extrapolate")(xn).astype(np.float32)


def add_cadence_fft_features(f: dict[str, float], kpts: np.ndarray, fps: float) -> None:
    dt = 1.0 / max(float(fps), 1.0)
    for side, ankle_idx in [("L", 15), ("R", 16)]:
        y = kpts[:, ankle_idx, 1].astype(np.float32)
        yc = y - np.mean(y)
        if len(yc) > 20:
            ac = np.correlate(yc, yc, mode="full")[len(yc) - 1 :]
            ac = ac / (float(ac[0]) + 1e-8)
            peaks = [
                j
                for j in range(1, len(ac) - 1)
                if ac[j] > ac[j - 1] and ac[j] > ac[j + 1] and ac[j] > 0.2
            ]
            if peaks:
                period = peaks[0] * dt
                f[f"{side}_cad"] = float(60.0 / period) if period > 0 else 0.0
                f[f"{side}_step_t"] = float(period)
                f[f"{side}_step_ac"] = float(ac[peaks[0]])

            fft = np.abs(np.fft.rfft(yc))
            freqs = np.fft.rfftfreq(len(yc), d=dt)
            if len(fft) > 1:
                dom = int(np.argmax(fft[1:]) + 1)
                f[f"{side}_dom_freq"] = float(freqs[dom])
                f[f"{side}_dom_amp"] = float(fft[dom] / max(len(yc), 1))
                mask = (freqs >= 0.5) & (freqs <= 2.0)
                f[f"{side}_walk_power"] = float(np.sum(fft[mask] ** 2) / (np.sum(fft**2) + 1e-8))


def add_phase_features(
    f: dict[str, float],
    kpts: np.ndarray,
    knee: np.ndarray,
    hip: np.ndarray,
    ankle: np.ndarray,
    fps: float,
) -> None:
    dt = 1.0 / max(float(fps), 1.0)
    if len(kpts) < 12:
        return
    vel = np.diff(kpts, axis=0) / dt
    for side, ankle_idx, knee_arr, hip_arr, ankle_arr in [
        ("L", 15, knee["L"], hip["L"], ankle["L"]),
        ("R", 16, knee["R"], hip["R"], ankle["R"]),
    ]:
        speed = np.linalg.norm(vel[:, ankle_idx], axis=-1)
        if len(speed) < 5:
            continue
        speed = np.r_[speed, speed[-1]]
        lo = np.percentile(speed, 40)
        hi = np.percentile(speed, 60)
        stance = speed <= lo
        swing = speed >= hi
        for phase_name, mask in [("stance", stance), ("swing", swing)]:
            if int(mask.sum()) < 3:
                continue
            f.update(stats(knee_arr[mask], f"{side}_{phase_name}_knee"))
            f.update(stats(hip_arr[mask], f"{side}_{phase_name}_hip"))
            f.update(stats(ankle_arr[mask], f"{side}_{phase_name}_ankle"))


def extract_features(kpts: np.ndarray | None, fps: float = 30.0) -> dict[str, float] | None:
    if kpts is None or len(kpts) < 10:
        return None
    kpts = np.asarray(kpts, dtype=np.float32)
    n = len(kpts)
    f: dict[str, float] = {"n_frames": float(n), "fps": float(fps)}

    hip_center = (kpts[:, 11] + kpts[:, 12]) / 2.0
    ankle_mid = (kpts[:, 15] + kpts[:, 16]) / 2.0
    body_height = float(np.nanmean(np.linalg.norm(kpts[:, 0] - ankle_mid, axis=-1))) + 1e-8
    norm = (kpts - hip_center[:, None, :]) / body_height

    angle_defs = {
        "Lkn": (11, 13, 15),
        "Rkn": (12, 14, 16),
        "Lhp": (5, 11, 13),
        "Rhp": (6, 12, 14),
        "Lak": (13, 15, 17),
        "Rak": (14, 16, 20),
        "Lakh": (13, 15, 19),
        "Rakh": (14, 16, 22),
    }
    angle_series: dict[str, np.ndarray] = {}
    for name, (a, b, c) in angle_defs.items():
        arr = ang3(kpts[:, a], kpts[:, b], kpts[:, c])
        angle_series[name] = arr
        f.update(stats(arr, name))

    shoulder_mid = (kpts[:, 5] + kpts[:, 6]) / 2.0
    hip_mid = (kpts[:, 11] + kpts[:, 12]) / 2.0
    trunk = np.degrees(np.arctan2((shoulder_mid - hip_mid)[:, 0], -(shoulder_mid - hip_mid)[:, 1] + 1e-8))
    f.update(stats(trunk, "trunk"))
    f.update(stats(kpts[:, 5, 1] - kpts[:, 6, 1], "shoulder_tilt_y"))
    f.update(stats(kpts[:, 11, 1] - kpts[:, 12, 1], "pelvis_tilt_y"))
    f.update(stats(np.abs(kpts[:, 5, 0] - kpts[:, 6, 0]), "shoulder_width"))
    f.update(stats(np.abs(kpts[:, 11, 0] - kpts[:, 12, 0]), "hip_width"))
    step_width = np.abs(kpts[:, 15, 0] - kpts[:, 16, 0])
    f.update(stats(step_width, "step_width"))
    f["step_width_cv"] = float(np.std(step_width) / (np.mean(step_width) + 1e-8))

    for side, ankle_idx, knee_idx, hip_idx, toe_idx, heel_idx in [
        ("L", 15, 13, 11, 17, 19),
        ("R", 16, 14, 12, 20, 22),
    ]:
        f.update(stats(kpts[:, ankle_idx, 1], f"{side}_ankle_y"))
        f.update(stats(kpts[:, ankle_idx, 0], f"{side}_ankle_x"))
        thigh = np.linalg.norm(kpts[:, hip_idx] - kpts[:, knee_idx], axis=-1)
        shank = np.linalg.norm(kpts[:, knee_idx] - kpts[:, ankle_idx], axis=-1)
        f.update(stats(thigh / (shank + 1e-8), f"{side}_thigh_shank"))
        knee_ankle_x = (kpts[:, knee_idx, 0] - kpts[:, hip_idx, 0]) - 0.5 * (kpts[:, ankle_idx, 0] - kpts[:, hip_idx, 0])
        f.update(stats(knee_ankle_x, f"{side}_knee_ankle_x"))
        toe_heel = kpts[:, toe_idx] - kpts[:, heel_idx]
        foot_angle = np.degrees(np.arctan2(toe_heel[:, 1], toe_heel[:, 0] + 1e-8))
        f.update(stats(foot_angle, f"{side}_foot_angle"))
        f.update(stats(kpts[:, toe_idx, 1] - kpts[:, heel_idx, 1], f"{side}_toe_heel_y"))
        f.update(stats(kpts[:, toe_idx, 0] - kpts[:, heel_idx, 0], f"{side}_toe_heel_x"))

    f["ankle_y_range_asym"] = abs(f.get("L_ankle_y_r", 0.0) - f.get("R_ankle_y_r", 0.0))
    f["step_time_asym"] = abs(f.get("L_step_t", 0.0) - f.get("R_step_t", 0.0))

    if n >= 3:
        dt = 1.0 / max(float(fps), 1.0)
        vel = np.diff(kpts, axis=0) / dt
        for name, idx in [("La", 15), ("Ra", 16), ("Lk", 13), ("Rk", 14), ("Lt", 17), ("Rt", 20)]:
            speed = np.linalg.norm(vel[:, idx], axis=-1)
            f.update(stats(speed, f"{name}_speed"))
            if len(speed) > 1:
                f[f"{name}_acc_abs_m"] = float(np.mean(np.abs(np.diff(speed) / dt)))

    add_cadence_fft_features(f, kpts, fps)

    for idx in GAIT_IDXS:
        f[f"n{idx}_xm"] = float(np.mean(norm[:, idx, 0]))
        f[f"n{idx}_ym"] = float(np.mean(norm[:, idx, 1]))
        f[f"n{idx}_xs"] = float(np.std(norm[:, idx, 0]))
        f[f"n{idx}_ys"] = float(np.std(norm[:, idx, 1]))
        f[f"n{idx}_xr"] = float(np.ptp(norm[:, idx, 0]))
        f[f"n{idx}_yr"] = float(np.ptp(norm[:, idx, 1]))

    for left, right, out_name in [
        ("Lkn", "Rkn", "knee"),
        ("Lhp", "Rhp", "hip"),
        ("Lak", "Rak", "ankle"),
        ("Lakh", "Rakh", "ankle_heel"),
    ]:
        for suffix in ["_m", "_s", "_r", "_p25", "_p75"]:
            lk = f"{left}{suffix}"
            rk = f"{right}{suffix}"
            if lk in f and rk in f:
                f[f"sym_{out_name}{suffix}"] = abs(f[lk] - f[rk]) / (abs(f[lk]) + abs(f[rk]) + 1e-8)

    f["cc_knee_y"] = safe_corr(kpts[:, 13, 1], kpts[:, 14, 1])
    f["cc_ankle_y"] = safe_corr(kpts[:, 15, 1], kpts[:, 16, 1])
    f["cc_ankle_x"] = safe_corr(kpts[:, 15, 0], kpts[:, 16, 0])

    head_y = kpts[:, 0, 1]
    f.update(stats(head_y, "head_y"))
    f.update(stats(kpts[:, 0, 0], "head_x"))
    f["head_bob_over_body"] = float(np.ptp(head_y) / body_height)
    f["head_sway_over_body"] = float(np.ptp(kpts[:, 0, 0]) / body_height)

    q1 = int(max(0, min(n - 1, round(0.15 * n))))
    q2 = int(max(q1 + 1, min(n, round(0.25 * n))))
    mid = slice(q1, q2)
    for name in ["Lkn", "Rkn", "Lhp", "Rhp", "Lak", "Rak"]:
        f.update(stats(angle_series[name][mid], f"midstance_{name}"))

    add_phase_features(
        f,
        kpts,
        knee={"L": angle_series["Lkn"], "R": angle_series["Rkn"]},
        hip={"L": angle_series["Lhp"], "R": angle_series["Rhp"]},
        ankle={"L": angle_series["Lak"], "R": angle_series["Rak"]},
        fps=fps,
    )

    for key, value in list(f.items()):
        if not np.isfinite(value):
            f[key] = 0.0
    return f


def build_features(patient_files: dict[int, dict[str, list[Path]]], patient_ids: list[int]) -> dict[int, dict[str, float]]:
    features: dict[int, dict[str, float]] = {}
    total = len(patient_ids)
    for i, pid in enumerate(sorted(patient_ids), start=1):
        per_view = []
        row: dict[str, float] = {}
        for view in VIEWS:
            files = patient_files.get(pid, {}).get(view, [])
            if not files:
                continue
            kpts, fps = load_kpts(files)
            ft = extract_features(kpts, fps)
            if ft is None:
                continue
            per_view.append(ft)
            for k, v in ft.items():
                row[f"{view}__{k}"] = float(v)
        if per_view:
            keys = sorted(set().union(*[set(d.keys()) for d in per_view]))
            for key in keys:
                vals = np.array([d[key] for d in per_view if key in d and np.isfinite(d[key])], dtype=np.float32)
                if vals.size:
                    row[f"avg__{key}"] = float(np.mean(vals))
                    if vals.size >= 2:
                        row[f"viewstd__{key}"] = float(np.std(vals))
            features[pid] = row
        if i % 10 == 0 or i == total:
            log(f"Built CGC features {i}/{total}; usable={len(features)}")
    return features


# =========================
# Track 1 model
# =========================

def make_matrix(pids: list[int], features: dict[int, dict[str, float]], feature_names: list[str] | None = None) -> tuple[np.ndarray, list[int], list[str]]:
    valid = [int(p) for p in pids if int(p) in features]
    if not valid:
        raise ValueError("No patient features available")
    if feature_names is None:
        feature_names = sorted(set().union(*[set(features[p].keys()) for p in valid]))
    X = np.array([[features[p].get(name, 0.0) for name in feature_names] for p in valid], dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, valid, feature_names


def make_models(seed: int) -> list[tuple[str, MultiOutputClassifier]]:
    return [
        (
            "gbm",
            MultiOutputClassifier(
                GradientBoostingClassifier(
                    n_estimators=200,
                    max_depth=3,
                    learning_rate=0.05,
                    subsample=0.8,
                    min_samples_leaf=3,
                    random_state=seed,
                ),
                n_jobs=N_JOBS,
            ),
        ),
        (
            "et",
            MultiOutputClassifier(
                ExtraTreesClassifier(
                    n_estimators=500,
                    max_depth=4,
                    min_samples_leaf=3,
                    random_state=seed,
                    n_jobs=N_JOBS,
                ),
                n_jobs=1,
            ),
        ),
        (
            "rf",
            MultiOutputClassifier(
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=4,
                    min_samples_leaf=3,
                    random_state=seed,
                    n_jobs=N_JOBS,
                ),
                n_jobs=1,
            ),
        ),
    ]


def multioutput_positive_proba(model: MultiOutputClassifier, X: np.ndarray) -> np.ndarray:
    cols = []
    for est in model.estimators_:
        if hasattr(est, "predict_proba"):
            proba = est.predict_proba(X)
            classes = list(getattr(est, "classes_", []))
            if 1 in classes:
                cols.append(proba[:, classes.index(1)])
            elif len(classes) == 1:
                cols.append(np.ones(len(X), dtype=np.float32) if int(classes[0]) == 1 else np.zeros(len(X), dtype=np.float32))
            else:
                cols.append(np.zeros(len(X), dtype=np.float32))
        else:
            cols.append(est.predict(X).astype(np.float32))
    return np.vstack(cols).T.astype(np.float32)


def s1_metric(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    acc = float((y_true == y_pred).mean())
    rmse = float(np.sqrt(np.mean((y_pred.sum(axis=1) - y_true.sum(axis=1)) ** 2)))
    score = float((acc + 1.0 - rmse / 34.0) / 2.0)
    return {"accuracy": acc, "rmse": rmse, "score": score}


def split_iterator(X: np.ndarray, y: np.ndarray, seed: int):
    totals = y.sum(axis=1).astype(int)
    strat = np.clip(totals, 0, 6)
    counts = pd.Series(strat).value_counts()
    if len(counts) > 1 and int(counts.min()) >= N_SPLITS:
        return StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed).split(X, strat)
    return KFold(N_SPLITS, shuffle=True, random_state=seed).split(X)


def tune_thresholds(y: np.ndarray, prob: np.ndarray) -> np.ndarray:
    thresholds = np.full(prob.shape[1], 0.5, dtype=np.float32)
    for c in range(prob.shape[1]):
        best_acc = -1.0
        best_t = 0.5
        for t in np.arange(0.30, 0.701, 0.05):
            acc = accuracy_score(y[:, c], (prob[:, c] >= t).astype(int))
            if acc > best_acc:
                best_acc = float(acc)
                best_t = float(t)
        thresholds[c] = best_t
    return thresholds


def train_track1_ensemble(X_raw: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, list[tuple[str, MultiOutputClassifier]], VarianceThreshold, StandardScaler, dict[str, float]]:
    vt = VarianceThreshold(threshold=0.0)
    X_vt = vt.fit_transform(X_raw)
    scaler = StandardScaler()
    X = scaler.fit_transform(X_vt)
    log(f"Track1 matrix after variance filter: {X.shape}")

    oof = np.zeros_like(y, dtype=np.float32)
    counts = np.zeros(len(y), dtype=np.int32)
    start = time.time()
    for seed_index, seed in enumerate(CV_SEEDS, start=1):
        for fold, (tr, va) in enumerate(split_iterator(X, y, seed), start=1):
            fold_prob = np.zeros((len(va), y.shape[1]), dtype=np.float32)
            for name, model in make_models(seed):
                model.fit(X[tr], y[tr])
                fold_prob += multioutput_positive_proba(model, X[va]) / 3.0
            oof[va] += fold_prob
            counts[va] += 1
            fold_score = s1_metric(y[va], (fold_prob >= 0.5).astype(int))
            log(
                f"seed {seed_index}/{len(CV_SEEDS)} fold {fold}: "
                f"S1@0.5={fold_score['score']:.5f} acc={fold_score['accuracy']:.5f} rmse={fold_score['rmse']:.5f}"
            )
            gc.collect()
        log(f"Completed CV seed {seed_index}/{len(CV_SEEDS)} in {(time.time() - start) / 60.0:.1f} min")

    oof = oof / np.maximum(counts[:, None], 1)
    thresholds = tune_thresholds(y, oof)
    tuned = (oof >= thresholds[None, :]).astype(int)
    cv_score = s1_metric(y, tuned)
    log(f"Track1 OOF tuned: S1={cv_score['score']:.5f} acc={cv_score['accuracy']:.5f} rmse={cv_score['rmse']:.5f}")

    final_models = make_models(SEED)
    for name, model in final_models:
        log(f"Fitting final {name}")
        model.fit(X, y)
    return oof, thresholds, final_models, vt, scaler, cv_score


def predict_track1(final_models: list[tuple[str, MultiOutputClassifier]], vt: VarianceThreshold, scaler: StandardScaler, X_raw: np.ndarray, thresholds: np.ndarray, shift: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
    X = scaler.transform(vt.transform(X_raw))
    prob = np.zeros((len(X), len(LABEL_COLS)), dtype=np.float32)
    for name, model in final_models:
        prob += multioutput_positive_proba(model, X) / len(final_models)
    th = np.clip(thresholds + float(shift), 0.05, 0.95)
    pred = (prob >= th[None, :]).astype(int)
    return pred, prob


# =========================
# Submission
# =========================

def load_best_track2_rows() -> pd.DataFrame:
    candidates = [
        WORK_DIR / "submission_stage1best_t1_stage2_t2.csv",
        WORK_DIR / "submission_stage1_t1_lesspos_01250_stage2t2_recreated.csv",
        WORK_DIR / "submission_stage2_pose_tcn.csv",
        WORK_DIR / "submission.csv",
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if "ID" in df.columns and df["ID"].astype(str).str.startswith("track2-").any():
                log(f"Using Track2 rows from {path}")
                return df[df["ID"].astype(str).str.startswith("track2-")].copy()
    raise FileNotFoundError("No prior submission with Track2 rows found in /kaggle/working")


def load_current_best_for_compare() -> pd.DataFrame | None:
    for path in [
        WORK_DIR / "submission_stage1best_t1_stage2_t2.csv",
        WORK_DIR / "submission_stage1_t1_lesspos_01250_stage2t2_recreated.csv",
        WORK_DIR / "submission.csv",
    ]:
        if path.exists():
            return pd.read_csv(path)
    return None


def make_submission(t1_pred: np.ndarray, t2_rows: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    rows = []
    for pid, labels in zip(T1_TEST_IDS, t1_pred.astype(int)):
        labels = labels.tolist()
        rows.append([f"track1-{pid}", *labels[:17], *labels[17:], int(sum(labels)), -1, -1])
    t1_df = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)
    out = pd.concat([t1_df, t2_rows[SUBMISSION_COLUMNS]], ignore_index=True)
    out = out[SUBMISSION_COLUMNS]
    assert out.shape == (25, 38), out.shape
    assert list(out["ID"][:16]) == [f"track1-{p}" for p in T1_TEST_IDS]
    assert list(out["ID"][16:]) == [f"track2-{p}" for p in T2_TEST_IDS]
    out.to_csv(out_path, index=False)
    log(f"Wrote {out_path}")
    return out


def compare_to_current_best(candidate: pd.DataFrame, current: pd.DataFrame | None) -> dict[str, Any]:
    if current is None:
        return {}
    cand = candidate[candidate["ID"].astype(str).str.startswith("track1-")].reset_index(drop=True)
    best = current[current["ID"].astype(str).str.startswith("track1-")].reset_index(drop=True)
    if len(cand) != len(best):
        return {}
    diff = cand[LABEL_COLS].astype(int).values - best[LABEL_COLS].astype(int).values
    changed = np.argwhere(diff != 0)
    by_patient: dict[str, int] = {}
    for row_idx, col_idx in changed:
        pid = str(cand.loc[row_idx, "ID"])
        by_patient[pid] = by_patient.get(pid, 0) + 1
    return {
        "changed_labels_vs_current_best": int(len(changed)),
        "changes_by_patient": by_patient,
        "candidate_totals": dict(zip(cand["ID"].tolist(), cand["Total"].astype(int).tolist())),
        "current_best_totals": dict(zip(best["ID"].tolist(), best["Total"].astype(int).tolist())),
    }


def main() -> None:
    start = time.time()
    seed_everything()
    ensure_dirs()

    section("Stage 3 CGC Feature Ensemble")
    data_root = find_data_root()
    log(f"RUN_ID={RUN_ID}")
    log(f"PATH1={PATH1}")
    log(f"PATH2={PATH2}")
    log(f"DATA_ROOT={data_root}")
    log(f"WORK_DIR={WORK_DIR}")

    if not PATH1.exists():
        raise FileNotFoundError(PATH1)
    if not data_root.exists():
        raise FileNotFoundError(data_root)

    config = {
        "run_id": RUN_ID,
        "seed": SEED,
        "max_frames_per_view": MAX_FRAMES_PER_VIEW,
        "cv_seeds": CV_SEEDS,
        "n_splits": N_SPLITS,
        "path1": str(PATH1),
        "path2": str(PATH2),
        "data_root": str(data_root),
    }
    dump_json(config, VERSION_DIR / "run_config.json")

    section("Load labels and build patient files")
    track1_df = flatten_track1(load_json(PATH1))
    patient_files = build_patient_files(data_root)
    all_pids = sorted(set(track1_df["patient_id"].astype(int).tolist()) | set(T1_TEST_IDS) | set(T2_TEST_IDS))
    log(f"Track1 train={track1_df.shape}, patient_file_count={len(patient_files)}, feature_pids={len(all_pids)}")
    track1_df.to_csv(TABLE_DIR / "track1_train_flat.csv", index=False)

    section("Extract CGC-inspired features")
    features = build_features(patient_files, all_pids)
    feature_dump = pd.DataFrame([{"patient_id": pid, **features[pid]} for pid in sorted(features)])
    feature_dump.to_csv(TABLE_DIR / "stage3_patient_features.csv", index=False)
    log(f"Feature table: {feature_dump.shape}")

    section("Train Track 1 ensemble")
    X_raw, train_pids, feature_names = make_matrix(track1_df["patient_id"].astype(int).tolist(), features)
    y_df = track1_df.set_index("patient_id")
    y = np.array([[int(y_df.loc[pid, col]) for col in LABEL_COLS] for pid in train_pids], dtype=np.int32)
    log(f"Track1 X_raw={X_raw.shape}, y={y.shape}")

    oof, thresholds, final_models, vt, scaler, cv_score = train_track1_ensemble(X_raw, y)
    pd.DataFrame(oof, columns=LABEL_COLS).assign(patient_id=train_pids).to_csv(TABLE_DIR / "stage3_track1_oof_prob.csv", index=False)
    pd.DataFrame({"label": LABEL_COLS, "threshold": thresholds}).to_csv(TABLE_DIR / "stage3_track1_thresholds.csv", index=False)
    with (VERSION_DIR / "feature_names.txt").open("w", encoding="utf-8") as f:
        for name in feature_names:
            f.write(name + "\n")

    if joblib is not None:
        joblib.dump({"models": final_models, "vt": vt, "scaler": scaler, "thresholds": thresholds, "feature_names": feature_names}, MODEL_DIR / "track1_stage3_cgc.joblib")
        log(f"Saved model bundle to {MODEL_DIR / 'track1_stage3_cgc.joblib'}")

    section("Create submissions with Stage 2 Track 2")
    X_test_raw, test_pids, _ = make_matrix(T1_TEST_IDS, features, feature_names)
    if test_pids != T1_TEST_IDS:
        raise ValueError(f"Missing test features. Got {test_pids}, expected {T1_TEST_IDS}")
    t2_rows = load_best_track2_rows()
    current_best = load_current_best_for_compare()

    outputs: dict[str, dict[str, Any]] = {}
    for shift, name in [
        (0.0, "submission_stage3_cgc_t1_stage2t2.csv"),
        (0.05, "submission_stage3_cgc_t1_stage2t2_lesspos005.csv"),
        (-0.05, "submission_stage3_cgc_t1_stage2t2_morepos005.csv"),
    ]:
        pred, prob = predict_track1(final_models, vt, scaler, X_test_raw, thresholds, shift=shift)
        out = make_submission(pred, t2_rows, WORK_DIR / name)
        info = compare_to_current_best(out, current_best)
        outputs[name] = {
            "threshold_shift": shift,
            "track1_totals": dict(zip([f"track1-{p}" for p in T1_TEST_IDS], pred.sum(axis=1).astype(int).tolist())),
            **info,
        }
        print("\nCandidate:", name, flush=True)
        print("threshold_shift:", shift, flush=True)
        print("Track1 totals:", outputs[name]["track1_totals"], flush=True)
        if info:
            print("changed_labels_vs_current_best:", info["changed_labels_vs_current_best"], flush=True)
            print("changes_by_patient:", info["changes_by_patient"], flush=True)

    summary = {
        "run_id": RUN_ID,
        "elapsed_minutes": (time.time() - start) / 60.0,
        "cv": {"track1": cv_score},
        "feature_table": str(TABLE_DIR / "stage3_patient_features.csv"),
        "outputs": outputs,
        "recommendation": (
            "Submit submission_stage3_cgc_t1_stage2t2.csv first only if Track1 OOF S1 is clearly above "
            "the old Stage1 OOF 0.7196 and changed labels are not extreme. Keep current best selected otherwise."
        ),
    }
    dump_json(summary, VERSION_DIR / "run_summary.json")
    log(f"Saved summary to {VERSION_DIR / 'run_summary.json'}")
    log(f"Elapsed minutes: {(time.time() - start) / 60.0:.2f}")
    section("Done")


if __name__ == "__main__":
    main()



Stage 3 CGC Feature Ensemble
[21:56:49] RUN_ID=stage3_cgc_20260428_215649
[21:56:49] PATH1=/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track1_train.json
[21:56:49] PATH2=/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track2_train.json
[21:56:49] DATA_ROOT=/kaggle/input/datasets/ibrahimqasimi/cvpr-2026-main-data/dataset
[21:56:49] WORK_DIR=/kaggle/working

Load labels and build patient files
[22:07:20] Indexed 250/1185 sequence dirs
[22:07:22] Indexed 500/1185 sequence dirs
[22:07:24] Indexed 750/1185 sequence dirs
[22:07:25] Indexed 1000/1185 sequence dirs
[22:07:26] Track1 train=(94, 36), patient_file_count=110, feature_pids=110

Extract CGC-inspired features
[22:07:52] Built CGC features 10/110; usable=10
[22:08:43] Built CGC features 20/110; usable=20
[22:09:57] Built CGC features 30/110; usable=30
[22:11:25] Built CGC features 40/110; usable=40
[22:13:09] Built CGC features 50/110; usable=50
[22:14:43] Built CGC features 60/110; usabl